In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Enter your student information
STUDENT_ID = "TUPM-26-1129"  
LAST_NAME = "Espejo"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(int(digit) for digit in id_digits)
NAME_LENGTH = len(LAST_NAME)

# Keep this formula consistent with the main laboratory activity.
RANDOM_SEED = ID_SUM + SEED_DIGIT + NAME_LENGTH

print("STUDENT CONFIGURATION")
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)


STUDENT CONFIGURATION
Seed Digit: 9
ID Sum: 21
Name Length: 6
Random Seed: 36


In [2]:

# ============================================================
# CELL 2: LOAD THE SECOM DATASET
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# Find the project folder automatically
CURRENT_DIR = Path.cwd()

PROJECT_DIR = None

for folder in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (folder / "data" / "secom.data").exists():
        PROJECT_DIR = folder
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "SECOM dataset not found! Check if secom.data "
        "is inside the data folder."
    )

# Set dataset location
DATA_PATH = PROJECT_DIR / "data" / "secom.data"

# Set output folder
OUTPUT_DIR = (
    PROJECT_DIR /
    "Exercise_1_Measurement_Quality" /
    "output"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
data = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None
)

# Assign feature names
data.columns = [
    f"Feature_{i+1}" for i in range(data.shape[1])
]

# Convert to NumPy matrix
measurement_matrix = data.to_numpy(dtype=float)

# Display dataset information
print("DATASET SUCCESSFULLY LOADED!")
print("-----------------------------------")
print("Dataset Location:", DATA_PATH)
print("Number of Records:", data.shape[0])
print("Number of Features:", data.shape[1])
print("Matrix Shape:", measurement_matrix.shape)

# Display first five records
data.head()


DATASET SUCCESSFULLY LOADED!
-----------------------------------
Dataset Location: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\data\secom.data
Number of Records: 1567
Number of Features: 590
Matrix Shape: (1567, 590)


,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,...,Feature_581,Feature_582,Feature_583,Feature_584,Feature_585,Feature_586,Feature_587,Feature_588,Feature_589,Feature_590
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [3]:
# Count missing values and observed values in each feature
missing_counts = np.isnan(measurement_matrix).sum(axis=0)
observed_counts = np.sum(~np.isnan(measurement_matrix), axis=0)

# Count distinct non-missing values in each feature
unique_counts = np.array([
    np.unique(measurement_matrix[~np.isnan(measurement_matrix[:, i]), i]).size
    for i in range(measurement_matrix.shape[1])
])

# A usable feature has at least one observed value and is not constant
usable_mask = (observed_counts > 0) & (unique_counts > 1)
usable_features = data.columns[usable_mask].to_numpy()
unusable_features = data.columns[~usable_mask].to_numpy()

# Missing percentage for every original feature
all_missing_percentages = (missing_counts / len(data)) * 100

print("FEATURE QUALITY CHECK")
print("Total features:", len(data.columns))
print("Usable features:", len(usable_features))
print("Unusable features (all missing or constant):", len(unusable_features))
print("Features with missing values:", int((missing_counts > 0).sum()))


FEATURE QUALITY CHECK
Total features: 590
Usable features: 474
Unusable features (all missing or constant): 116
Features with missing values: 538


In [4]:
# Requirement: select 5 + (SEED_DIGIT % 3) usable features
number_selected = 5 + (SEED_DIGIT % 3)

if len(usable_features) < number_selected:
    raise ValueError(
        f"Only {len(usable_features)} usable features are available; "
        f"{number_selected} are required."
    )

rng = np.random.default_rng(RANDOM_SEED)
selected_features = list(
    rng.choice(usable_features, size=number_selected, replace=False)
)

selected_data = data[selected_features].to_numpy(dtype=float)

print("SELECTED FEATURES")
for feature in selected_features:
    print("-", feature)
print("Number selected:", len(selected_features))


SELECTED FEATURES
- Feature_199
- Feature_219
- Feature_92
- Feature_207
- Feature_226
Number selected: 5


In [5]:
# NumPy calculations ignore missing values (NaN)
feature_means = np.nanmean(selected_data, axis=0)
feature_medians = np.nanmedian(selected_data, axis=0)
feature_std = np.nanstd(selected_data, axis=0, ddof=1)
missing_percentages = np.isnan(selected_data).mean(axis=0) * 100

statistics = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_Std_Dev": feature_std,
    "Missing_Percentage": missing_percentages
})

print("FEATURE-WISE STATISTICS")
statistics


FEATURE-WISE STATISTICS


,Feature,Mean,Median,Sample_Std_Dev,Missing_Percentage
0,Feature_199,0.557359,0.42400,0.537705,0.382897
1,Feature_219,3.771465,3.63075,1.170436,0.063816
2,Feature_92,0.002440,0.00000,0.087683,0.382897
3,Feature_207,0.001281,0.00000,0.050621,0.382897
4,Feature_226,1041.056588,967.29980,433.170076,3.254627


In [6]:
# Median Absolute Deviation (MAD)
absolute_deviations = np.abs(selected_data - feature_medians)
mad_values = np.nanmedian(absolute_deviations, axis=0)

# Initialize robust z-scores. A zero-MAD feature is safely assigned 0
# because division by zero cannot produce a meaningful robust z-score.
robust_z = np.zeros_like(selected_data, dtype=float)

valid_mad = np.isfinite(mad_values) & (mad_values > 0)

robust_z[:, valid_mad] = (
    0.6745
    * (selected_data[:, valid_mad] - feature_medians[valid_mad])
    / mad_values[valid_mad]
)

# Keep missing measurements as NaN rather than treating them as outliers.
robust_z[np.isnan(selected_data)] = np.nan

mad_summary = pd.DataFrame({
    "Feature": selected_features,
    "Median": feature_medians,
    "MAD": mad_values,
    "MAD_Zero_or_Invalid": ~valid_mad
})

mad_summary


,Feature,Median,MAD,MAD_Zero_or_Invalid
0,Feature_199,0.42400,0.16200,False
1,Feature_219,3.63075,0.73045,False
2,Feature_92,0.00000,0.04630,False
3,Feature_207,0.00000,0.00000,True
4,Feature_226,967.29980,271.95020,False


In [7]:
# Flag measurements whose absolute robust z-score is greater than 3.5
flagged_matrix = np.abs(robust_z) > 3.5
flagged_matrix = np.nan_to_num(flagged_matrix, nan=False).astype(bool)

# Number and percentage of flagged measurements in each selected feature
flagged_counts = flagged_matrix.sum(axis=0)
flagged_percentages = (flagged_counts / len(data)) * 100

# Total flagged measurement values and unique records with at least one flag
total_flagged_measurements = int(flagged_counts.sum())
unique_flagged_records = int(flagged_matrix.any(axis=1).sum())

print("OUTLIER DETECTION")
print("Total flagged measurement values:", total_flagged_measurements)
print("Unique records with at least one flagged measurement:", unique_flagged_records)

outlier_summary = pd.DataFrame({
    "Feature": selected_features,
    "Flagged_Observations": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})
outlier_summary


OUTLIER DETECTION
Total flagged measurement values: 100
Unique records with at least one flagged measurement: 97


,Feature,Flagged_Observations,Flagged_Percentage
0,Feature_199,49,3.126994
1,Feature_219,9,0.574346
2,Feature_92,25,1.595405
3,Feature_207,0,0.000000
4,Feature_226,17,1.084876


In [8]:
summary = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_Std_Dev": feature_std,
    "Missing_Percentage": missing_percentages,
    "MAD": mad_values,
    "Flagged_Observations": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

# Rank from the highest flagged percentage to the lowest
summary = summary.sort_values(
    by="Flagged_Percentage",
    ascending=False
).reset_index(drop=True)

summary.insert(0, "Rank", range(1, len(summary) + 1))

output_file = OUTPUT_DIR / "measurement_quality.csv"
summary.to_csv(output_file, index=False)

print("RANKED FEATURE SUMMARY")
display(summary)
print("CSV saved to:", output_file)


RANKED FEATURE SUMMARY


,Rank,Feature,Mean,Median,Sample_Std_Dev,Missing_Percentage,MAD,Flagged_Observations,Flagged_Percentage
0,1,Feature_199,0.557359,0.42400,0.537705,0.382897,0.16200,49,3.126994
1,2,Feature_92,0.002440,0.00000,0.087683,0.382897,0.04630,25,1.595405
2,3,Feature_226,1041.056588,967.29980,433.170076,3.254627,271.95020,17,1.084876
3,4,Feature_219,3.771465,3.63075,1.170436,0.063816,0.73045,9,0.574346
4,5,Feature_207,0.001281,0.00000,0.050621,0.382897,0.00000,0,0.000000


CSV saved to: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_1_Measurement_Quality\output\measurement_quality.csv


In [9]:
# Recalculate the same statistics using Pandas
pandas_means = data[selected_features].mean().to_numpy()
pandas_medians = data[selected_features].median().to_numpy()
pandas_std = data[selected_features].std(ddof=1).to_numpy()
pandas_missing = data[selected_features].isna().mean().to_numpy() * 100

vectorized_verified = (
    np.allclose(feature_means, pandas_means, equal_nan=True)
    and np.allclose(feature_medians, pandas_medians, equal_nan=True)
    and np.allclose(feature_std, pandas_std, equal_nan=True)
    and np.allclose(missing_percentages, pandas_missing, equal_nan=True)
)

print("NumPy and Pandas statistics match:", vectorized_verified)


NumPy and Pandas statistics match: True


In [10]:
# Highest missing percentage is evaluated across ALL original features.
highest_missing_index_all = int(np.argmax(all_missing_percentages))
highest_missing_feature_all = data.columns[highest_missing_index_all]
highest_missing_percentage_all = float(all_missing_percentages[highest_missing_index_all])

# Highest flagged percentage is evaluated among the selected features.
highest_flagged_index = int(np.argmax(flagged_percentages))
highest_flagged_feature = selected_features[highest_flagged_index]
highest_flagged_percentage = float(flagged_percentages[highest_flagged_index])

assessment = pd.DataFrame({
    "Output Parameter": [
        "Selected Features",
        "Highest Missing Percentage",
        "Feature with Highest Missing Percentage",
        "Feature with Highest Flagged Percentage",
        "Total Flagged Observations",
        "Vectorized Results Verified"
    ],
    "Observed Value": [
        number_selected,
        f"{highest_missing_percentage_all:.4f}%",
        highest_missing_feature_all,
        f"{highest_flagged_feature} ({highest_flagged_percentage:.4f}%)",
        total_flagged_measurements,
        bool(vectorized_verified)
    ]
})

print("=" * 70)
print("EXERCISE 1: ASSESSMENT DATA")
print("=" * 70)
display(assessment)

print("Required output: measurement_quality.csv")
print("Saved location:", output_file)
print("=" * 70)


EXERCISE 1: ASSESSMENT DATA


,Output Parameter,Observed Value
0,Selected Features,5
1,Highest Missing Percentage,91.1934%
2,Feature with Highest Missing Percentage,Feature_158
3,Feature with Highest Flagged Percentage,Feature_199 (3.1270%)
4,Total Flagged Observations,100
5,Vectorized Results Verified,True


Required output: measurement_quality.csv
Saved location: c:\Users\avega\OneDrive\Documents\ESPEJO\Lab7_NumPy_Pandas_Espejo\Lab7_NumPy_Pandas_Additional_Exercises\Exercise_1_Measurement_Quality\output\measurement_quality.csv
